# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5, (2, 3) - извлечен элемент на позиции [1, 2] (это 5), и выведена форма массива (2, 3)
2. 1, 8 - первый индекс максимального элемента равен 1, а сам максимум равен 8
3. 99 - срез в NumPy создает представление (view), поэтому изменение b[0] = 99 изменило и исходный массив a[0]
4. 2, 0.5 - элементов больше 3 ровно 2 штуки (сумма = 2), а их доля от общего числа составляет 50% (среднее = 0.5)
5. (2, 3), 0.2 - размер матрицы из нулей равен (2, 3), а второй элемент линейного пространства от 0 до 1 (при 5 точках) равен 0.2

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [4]:
# База
# 1
print(f"Позиция: {amount.argmax()}, Сумма: {amount[amount.argmax()]}, Клиент: {cust[amount.argmax()]}, Город: {city[amount.argmax()]}")

Позиция: 1001, Сумма: 9999990.0, Клиент: TEST, Город: Алматы


In [5]:
# 2
amount_paid = amount[paid]
amount_unpaid = amount[~paid]
print(f"Выручка оплаченных заказов: {amount_paid.sum()}, сумма неоплаченных: {amount_unpaid.sum()}")

Выручка оплаченных заказов: 101909360.0, сумма неоплаченных: 7974000.0


In [6]:
# 3
print(f"Количество заказов в Шымкенте: {(city == 'Шымкент').sum()}, доля от всех заказов: {(city == 'Шымкент').mean()}")

Количество заказов в Шымкенте: 804, доля от всех заказов: 0.1471718835804503


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [7]:
# Стандарт
# 4
amount_paid_more_50k = amount_paid[amount_paid > 50000]
print(f"Выручки магазин недополучила: {amount_paid_more_50k.sum() * 0.15}")

loss_without_test = amount[(amount > 50000) & paid & (cust != "TEST")]
print(f"Выручки магазин недополучила без заказов клиента TEST: {loss_without_test.sum() * 0.15}")

Выручки магазин недополучила: 5702422.5
Выручки магазин недополучила без заказов клиента TEST: 1202427.0


In [8]:
# 5
unique_cities = np.unique(city)
for cities in unique_cities:
    print(f"Город: {cities}, Доля заказов, доставленных за 1-2 дня: {((city == cities) & (days >= 1) & (days <= 2)).mean()}")

Город: Актобе, Доля заказов, доставленных за 1-2 дня: 0.0452132527915065
Город: Алматы, Доля заказов, доставленных за 1-2 дня: 0.25846604429800474
Город: Астана, Доля заказов, доставленных за 1-2 дня: 0.16163280248947465
Город: Атырау, Доля заказов, доставленных за 1-2 дня: 0.017206663005674538
Город: Караганда, Доля заказов, доставленных за 1-2 дня: 0.06681310635182135
Город: Шымкент, Доля заказов, доставленных за 1-2 дня: 0.043931905546403076


In [9]:
# 6
avg = amount[paid].mean()
print("Средний чек:", round(avg, 2))

Средний чек: 20780.86


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [13]:
# Челлендж
# 7
import time
start_time = time.time()
total_sum_for = 0
for i in range(len(amount)):
  if paid[i]:
    total_sum_for += amount[i]
time_for = time.time() - start_time

start_time = time.time()
total_sum_mask = amount[paid].sum()
time_mask = time.time() - start_time

print(f"Время с циклом for: {time_for: .6f} сек")
print(f"Время с маской NumPy: {time_mask: .6f} сек")

print(f"Mask faster than for {round(time_for / time_mask)} times")

Время с циклом for:  0.003842 сек
Время с маской NumPy:  0.000206 сек
Mask faster than for 19 times
